In [43]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
import adi
import binascii
import time

In [44]:
def text_to_bin(text):
    temp_bin_str = ''.join(format(ord(c), '07b') for c in text)
    bin_data = np.array([int(bit) for bit in temp_bin_str], dtype=int)
    return bin_data

def append_crc_to_frame(payload):
    byte_data = np.packbits(payload)  
    crc = binascii.crc_hqx(byte_data, 0xFFFF)  
    crc_bits = np.array(list(np.binary_repr(crc, width=16)), dtype=np.uint8)  
    return np.concatenate((payload, crc_bits))

def int_to_fixed_binary_array(n, bits=8):
    return [int(bit) for bit in format(n, f'0{bits}b')]

def bin_to_text(binary_str):
    decoded_bits = "".join(str(bit) for bit in binary_str)
    text = ""
    for i in range(0, len(decoded_bits), 7):
        byte = decoded_bits[i:i+7]
        if len(byte) == 7 :
            text += chr(int(byte, 2))
    return text

In [45]:
from transformers import GPT2Tokenizer, GPT2LMHeadModel
import torch
import string
from sentence_transformers import SentenceTransformer, util
import re
from collections import Counter
import numpy as np

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2")
model.eval()
model11 = SentenceTransformer('all-MiniLM-L6-v2')

In [46]:
with open('test_file_1.txt') as f:
  test_text = f.read()
text_words = re.findall(r"\w+|[^\w\s]", test_text, re.UNICODE)

no_seed = 50
seed_text = " ".join(text_words[:no_seed])
print('seed text: ', seed_text)
input_test_words = text_words[no_seed:]

def Transmitter_coding(seed_text, input_text):
    tx_words = []
    count_char = 0
    for k in range(len(input_text)):
        original_tokens = seed_text.split()
        actual = input_text[k]
        if len(actual) < 2:
            new_prompt = ' '.join(original_tokens[1:] + [actual.strip()])
            tx_words.append(actual)
            count_char+= len(actual)+1
        else:
            input_ids = tokenizer.encode(seed_text, return_tensors='pt')
            with torch.no_grad():
                output = model.generate(
                    input_ids,
                    max_new_tokens=1,
                    do_sample=False,
                    pad_token_id=tokenizer.eos_token_id)
            predicted_token = output[0][input_ids.shape[-1]:]
            pred = tokenizer.decode(predicted_token, skip_special_tokens=True).strip()
            pred_clean = re.sub(r"[^\w']+", "", pred)
            actual_clean = re.sub(r"[^\w']+", "", actual)
            embeddings = model11.encode([pred_clean, actual_clean], convert_to_tensor=True)
            similarity = util.cos_sim(embeddings[0], embeddings[1])
            if similarity.item() >= 0.7:
                new_prompt = ' '.join(original_tokens[1:] + [pred.strip()])
                tx_words.append('_')
                count_char+= 2
            else:
                new_prompt = ' '.join(original_tokens[1:] + [actual.strip()])
                tx_words.append(actual)
                count_char+= len(actual)+1
        # print(count_char)
        if count_char > 120:
            break
        seed_text = new_prompt
    tx_sentece = ' '.join(tx_words)
    return tx_sentece, seed_text, k


seed text:  In the digital age , privacy has become one of the most debated and critical issues . With every click , swipe , and search , individuals leave behind a trail of data . This data , often collected without explicit consent , is used by corporations , advertisers ,


In [47]:
from IPython.display import display, HTML

def show_message(text):
    display(HTML(f"""
    <div style="
        background-color:white;
        color:black;
        font-size:24px;
        padding:20px;
        border:2px solid black;
        border-radius:10px;
        width:300px;
        text-align:center;">
        {text}
    </div>
    """))

In [48]:
init = 0

frame_no = 0

no_seed = 50
seed_text = " ".join(text_words[:no_seed])
print('seed text: ', seed_text)

seed text:  In the digital age , privacy has become one of the most debated and critical issues . With every click , swipe , and search , individuals leave behind a trail of data . This data , often collected without explicit consent , is used by corporations , advertisers ,


In [49]:
while(1):
    test_text_words = input_test_words[init:init+40]
    # print(init, init+40)

    final_tx, seed_tx_next, count_words = Transmitter_coding(seed_text, test_text_words)
    # print('count words', count_words)
    init+= count_words + 1

    # print('new init', init)

    seed_text = seed_tx_next

    temp_text = final_tx

    # print('text length:', len(temp_text))

    append_chars = (145 - len(temp_text))*'*'

    text = "".join([temp_text, append_chars])


    message_bits = text_to_bin(text)
    # print('\nmessage bits length =', len(message_bits))



    payload_size =  len(message_bits)
    # print(payload_size)
    n_bits = payload_size + 16 + 10

    frame_id = int_to_fixed_binary_array(frame_no, 10)
    payload = np.concatenate((frame_id, message_bits[:payload_size]))
    data_bits = append_crc_to_frame(payload)

    bpsk_symbols = np.array([-1 if bit == 0 else 1 for bit in data_bits])
    barker_code = np.array([1, 1, 1, 1, 1, -1, -1, 1, 1, -1, 1, -1, 1])
    barker_appended_bpsk = np.concatenate((barker_code,  bpsk_symbols))

    sps = 8
    zero_padded = np.zeros(len(barker_appended_bpsk) * sps)
    zero_padded[::sps] = barker_appended_bpsk

    num_taps = 101
    beta = 0.34
    Ts = sps
    t = np.arange(num_taps) - (num_taps-1)//2
    rc_pulse = np.sinc(t/Ts) * np.cos(np.pi*beta*t/Ts) / (1 - (2*beta*t/Ts)**2)
    ps_conv_output = np.convolve(zero_padded, rc_pulse, mode = 'full')
    pulse_shaped = ps_conv_output[num_taps//2: -1-num_taps//2]

    tx_signal = pulse_shaped * (2**14)


    sample_rate = 10e6 # Hz
    carrier_freq = 985e6 # Hz
    num_samps = 100000 

    sdr = adi.Pluto("ip:192.168.2.1")
    sdr.sample_rate = int(sample_rate)

    sdr.tx_rf_bandwidth = int(sample_rate) 
    sdr.tx_lo = int(carrier_freq)
    sdr.tx_hardwaregain_chan0 = -15

    sdr.tx_cyclic_buffer = True 
    sdr.tx(tx_signal)  



    print('\ntransmission started:', frame_no)

    payload_bits = data_bits[10:payload_size+10]
    decoded_text = bin_to_text(payload_bits)
    # print(decoded_text, end = '')

    show_message(decoded_text[:-(145 - len(temp_text))])

    frame_no+= 1

    time.sleep(5)

ERROR: READ LINE: -104
ERROR: Unable to send command: Broken pipe (32)



transmission started: 0



transmission started: 1



transmission started: 2



transmission started: 3


...


transmission started: 9



transmission started: 10



transmission started: 11



transmission started: 12


KeyboardInterrupt: 